In [1]:
from IPython.display import clear_output
!pip install -q albumentations
clear_output()

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import numpy as np
import os
from glob import glob
from sklearn.model_selection import train_test_split

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [4]:
image_size = 224
latent_dim = 512
batch_size = 64
epochs = 50
learning_rate = 1e-3
train_split = 0.9  

In [5]:
transform = A.Compose([
    A.Resize(height=image_size, width=image_size),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

In [6]:
class ISICDataset(Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']
        return image

In [7]:
image_dir = '/kaggle/input/isic-2019/ISIC_2019_Training_Input/ISIC_2019_Training_Input'
image_paths = glob(os.path.join(image_dir, '*.jpg'))
train_paths, val_paths = train_test_split(image_paths, train_size=train_split, random_state=42)

In [8]:
train_dataset = ISICDataset(image_paths=train_paths, transform=transform)
val_dataset = ISICDataset(image_paths=val_paths, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [9]:
class Autoencoder(nn.Module):
    def __init__(self):
        super(Autoencoder, self).__init__()
        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1),  # 3x224x224 -> 32x112x112
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),  # -> 64x56x56
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),  # -> 128x28x28
            nn.ReLU(),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),  # -> 256x14x14
            nn.ReLU(),
            nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1),  # -> 512x7x7
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(512 * 7 * 7, latent_dim)
        )
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 512 * 7 * 7),
            nn.Unflatten(1, (512, 7, 7)),
            nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1),  # 512x7x7 -> 256x14x14
            nn.ReLU(),
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),  # -> 128x28x28
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),  # -> 64x56x56
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),  # -> 32x112x112
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1),  # -> 3x224x224
            nn.Tanh()
        )

    def forward(self, x):
        latent = self.encoder(x)
        reconstructed = self.decoder(latent)
        return reconstructed

In [10]:
model = Autoencoder().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [11]:
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for data in train_loader:
        img = data.to(device)  # No labels needed
        # Forward pass
        output = model(img)
        loss = criterion(output, img)
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Epoch [{epoch+1}/{epochs}], Loss: {total_loss / len(train_loader):.4f}')

    if (epoch + 1) % 10 == 0:
        checkpoint_path = f'autoencoder_skin_epoch_{epoch+1}.pth'
        torch.save(model.state_dict(), checkpoint_path)
        print(f'Saved checkpoint: {checkpoint_path}')

Epoch [1/50], Loss: 0.3558
Epoch [2/50], Loss: 0.2091
Epoch [3/50], Loss: 0.1970
Epoch [4/50], Loss: 0.1892
Epoch [5/50], Loss: 0.1856
Epoch [6/50], Loss: 0.1833
Epoch [7/50], Loss: 0.1819
Epoch [8/50], Loss: 0.1801
Epoch [9/50], Loss: 0.1796
Epoch [10/50], Loss: 0.1785
Saved checkpoint: autoencoder_skin_epoch_10.pth
Epoch [11/50], Loss: 0.1784
Epoch [12/50], Loss: 0.1773
Epoch [13/50], Loss: 0.1764
Epoch [14/50], Loss: 0.1762
Epoch [15/50], Loss: 0.1756
Epoch [16/50], Loss: 0.1756
Epoch [17/50], Loss: 0.1752
Epoch [18/50], Loss: 0.1748
Epoch [19/50], Loss: 0.1744
Epoch [20/50], Loss: 0.1741
Saved checkpoint: autoencoder_skin_epoch_20.pth
Epoch [21/50], Loss: 0.1744
Epoch [22/50], Loss: 0.1738
Epoch [23/50], Loss: 0.1734
Epoch [24/50], Loss: 0.1730
Epoch [25/50], Loss: 0.1732
Epoch [26/50], Loss: 0.1726
Epoch [27/50], Loss: 0.1725
Epoch [28/50], Loss: 0.1725
Epoch [29/50], Loss: 0.1723
Epoch [30/50], Loss: 0.1716
Saved checkpoint: autoencoder_skin_epoch_30.pth
Epoch [31/50], Loss: 0.17

In [12]:
torch.save(model, 'autoencoder_skin.pth')

In [13]:
# Compute reconstruction errors on validation set to set threshold
def get_reconstruction_errors(model, dataloader, criterion, device):
    model.eval()
    errors = []
    with torch.no_grad():
        for data in dataloader:
            img = data.to(device)
            output = model(img)
            loss = criterion(output, img)
            errors.append(loss.item())
    return errors

val_errors = get_reconstruction_errors(model, val_loader, criterion, device)
threshold = np.mean(val_errors) + 3 * np.std(val_errors)
print(f'Validation MSE - Mean: {np.mean(val_errors):.4f}, Std: {np.std(val_errors):.4f}, Threshold: {threshold:.4f}')

Validation MSE - Mean: 0.1710, Std: 0.0263, Threshold: 0.2500
